# OrCaScore — CT com e sem contraste e calcificações de referência

Visualização de duas aquisições do mesmo paciente: **CTAI** com contraste e
**CTI** sem contraste. O exemplo padrão é **TRV1P3**, que possui calcificações
anotadas. TRV1P1, usado anteriormente, possui uma referência válida toda zerada.

- MIPs axiais e fatias centrais das duas aquisições, em células separadas.
- CT sem contraste, rótulos de calcificação e sobreposição, lado a lado e também
  em figuras independentes, com recorte da região anotada.
- Superfície 3D opcional das calcificações de referência.

A orientação dos arquivos é preservada, **sem rotação, flip ou alinhamento ao
ImageCAS**. As aquisições CTAI e CTI não são registradas entre si. Os rótulos
pertencem à CTI e **não são sobrepostos à CTAI**.

A [documentação oficial do OrCaScore](https://orcascore.grand-challenge.org/Data/)
define 1 = LAD (inclui calcificações do tronco esquerdo), 2 = LCX e 3 = RCA.
São rótulos de calcificações atribuídas a cada artéria, não máscaras das artérias
completas. Neste banco local, os rótulos de referência estão no treino.

**Impressão:** PNG e TIFF LZW sem perdas a 600 dpi e PDF. As figuras independentes
usam 90 × 90 mm, sem títulos ou eixos. O DPI não aumenta a resolução anatômica
original. A configuração informa o diretório permanente de exportação.
Não é necessário executar segmentação ou usar GPU.

In [ ]:
# ruff: noqa: E402
import os
import sys
from pathlib import Path

import numpy as np

REPO_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "src").is_dir()
)
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from utils.project.datasets.ccta import (
    discover_orcascore_acquisitions,
    load_ccta_volume,
    read_mhd_header,
)
from utils.project.datasets.orcascore import (
    ORCASCORE_CALCIFICATION_LABEL_SPECS,
    load_orcascore_calcification_reference,
    orcascore_calcification_reference_path,
)
from utils.project.runtime.notebook_env import resolve_existing_path
from utils.utils.roi import mask_bounding_box_slices

## Configuração

Altere `EXAM_ID` e `SUBSET` para outro par. O diretório pode ser definido por
`ORCASCORE_BASE_PATH` ou descoberto nos discos montados. A janela HU é comum às
duas aquisições e às sobreposições. `OUTPUT_ROOT` é o destino das exportações;
`ORCASCORE_FIGURES_ROOT` permite configurá-lo no ambiente.

Na seção de calcificação, o padrão escolhe a fatia **com maior área anotada**,
para tornar a referência visível. `CALCIFICATION_SLICE_INDEX` permite escolher
um índice axial nativo, começando em zero. `CALCIFICATION_MARGIN_MM` define a
margem do recorte em torno das calcificações. Essa seção não representa a fatia
central do volume, que continua disponível nas células anteriores.

In [ ]:
# Para outro disco, exporte ORCASCORE_BASE_PATH antes de abrir o kernel.
# Sem variável, procura somente a estrutura conhecida nos discos montados.
MOUNT_ROOTS = (Path("/run/media"), Path("/media"), Path("/mnt"))
DATASET_CANDIDATES = sorted(
    {
        candidate
        for root in MOUNT_ROOTS
        for pattern in (
            "*/*/DatasetsCCTA/Orca_Score_Calcium",
            "*/DatasetsCCTA/Orca_Score_Calcium",
        )
        for candidate in root.glob(pattern)
        if candidate.is_dir()
    }
)
if not os.environ.get("ORCASCORE_BASE_PATH") and len(DATASET_CANDIDATES) > 1:
    raise ValueError("Mais de uma cópia do OrCaScore: defina ORCASCORE_BASE_PATH.")
ORCASCORE_PATH = resolve_existing_path(
    "ORCASCORE_BASE_PATH", DATASET_CANDIDATES, "OrCaScore"
)

EXAM_ID = "TRV1P3"
SUBSET = "train"
WINDOW_LEVEL_HU = 200.0
WINDOW_WIDTH_HU = 800.0
FIGURE_WIDTH_MM = 90.0
EXPORT_DPI = 600
PREVIEW_DPI = 120
OUTPUT_ROOT = Path(
    os.environ.get("ORCASCORE_FIGURES_ROOT")
    or REPO_ROOT / "output/figures/orcascore_calcification"
).expanduser()
CALCIFICATION_MARGIN_MM = 10.0
CALCIFICATION_SLICE_INDEX = None
CALCIFICATION_ALPHA = 0.5
SHOW_CALCIFICATION_3D = False

In [ ]:
inventory = discover_orcascore_acquisitions(ORCASCORE_PATH)
selected = inventory.loc[
    inventory["exam_id"].eq(EXAM_ID) & inventory["subset"].eq(SUBSET)
].copy()
if len(selected) != 2 or set(selected["acquisition"]) != {"contrast", "noncontrast"}:
    raise ValueError(f"Esperado exatamente um par CTAI/CTI para {SUBSET}/{EXAM_ID}.")
PAIR = {
    acquisition: selected.loc[selected["acquisition"].eq(acquisition)].iloc[0]
    for acquisition in ("contrast", "noncontrast")
}
if WINDOW_WIDTH_HU <= 0 or not np.isfinite([WINDOW_LEVEL_HU, WINDOW_WIDTH_HU]).all():
    raise ValueError("Janela HU inválida.")
if FIGURE_WIDTH_MM <= 0 or EXPORT_DPI <= 0:
    raise ValueError("Tamanho de impressão e DPI devem ser positivos.")
if (
    selected["reported_orientation"].nunique() != 1
    or selected["reported_orientation"].iloc[0] != "RAI"
):
    raise ValueError("Este temporário espera o par OrCaScore na orientação RAI.")
for record in PAIR.values():
    header = read_mhd_header(record["path"])
    transform = np.fromstring(
        header.get("TransformMatrix", "1 0 0 0 1 0 0 0 1"), sep=" "
    )
    if transform.size != 9 or not np.allclose(transform.reshape(3, 3), np.eye(3)):
        raise ValueError("Aquisição oblíqua: revise a orientação antes da figura.")
# Mesmo campo físico quadrado, centrado no FOV de cada aquisição; sem registrar volumes.
COMMON_FOV_MM = max(
    float(record[size]) * float(record[spacing])
    for record in PAIR.values()
    for size, spacing in (("size_x", "spacing_x_mm"), ("size_y", "spacing_y_mm"))
)
OUTPUT_DIR = OUTPUT_ROOT / f"{SUBSET}_{EXAM_ID}"
print(f"Par selecionado: {SUBSET}/{EXAM_ID} | MIP axial do volume completo")
for acquisition, record in PAIR.items():
    print(
        f"{acquisition}: {Path(record['path']).name}; "
        f"{int(record['slice_count'])} fatias; z={record['spacing_z_mm']:.3f} mm"
    )
print(f"Janela comum: WL={WINDOW_LEVEL_HU:g} / WW={WINDOW_WIDTH_HU:g} HU")
print(f"Exportação: {FIGURE_WIDTH_MM:g} mm, {EXPORT_DPI} dpi; {OUTPUT_DIR}")

In [ ]:
from eda.images.view.orcascore_calcification_views import (
    export_axial_mip,
    export_central_axial_slice,
)

AXIAL_EXPORT_CONFIG = {
    "pair": PAIR,
    "subset": SUBSET,
    "exam_id": EXAM_ID,
    "common_fov_mm": COMMON_FOV_MM,
    "window_level_hu": WINDOW_LEVEL_HU,
    "window_width_hu": WINDOW_WIDTH_HU,
    "figure_width_mm": FIGURE_WIDTH_MM,
    "preview_dpi": PREVIEW_DPI,
    "export_dpi": EXPORT_DPI,
    "output_dir": OUTPUT_DIR,
}

## Com contraste — CTAI

In [ ]:
export_axial_mip("contrast", **AXIAL_EXPORT_CONFIG)

## Sem contraste — CTI

In [ ]:
export_axial_mip("noncontrast", **AXIAL_EXPORT_CONFIG)

## Fatias axiais centrais

Cada aquisição usa a fatia de índice **`n_z // 2`**, começando em zero. Para um
número par de fatias, seleciona a segunda das duas centrais. CTAI e CTI têm
coberturas diferentes: os centros não representam necessariamente a mesma
posição anatômica. São mantidas a orientação nativa, a janela HU, a escala física
e a exportação em PNG/TIFF a 600 dpi e PDF.

### Fatia central com contraste — CTAI

In [ ]:
export_central_axial_slice("contrast", **AXIAL_EXPORT_CONFIG)

### Fatia central sem contraste — CTI

In [ ]:
export_central_axial_slice("noncontrast", **AXIAL_EXPORT_CONFIG)

## Calcificações na CT sem contraste — CT, rótulos e sobreposição

A referência `Reference_standard/<EXAM_ID>R.mhd` só é usada após validar a
geometria contra a CTI: dimensões, espaçamento, origem, direção e orientação.
Sem referência, as visualizações acima continuam disponíveis e esta seção
informa a ausência. Uma referência toda zerada é tratada como exame **sem
calcificações anotadas**, sem confundi-la com arquivo ausente.

O recorte usa a caixa das calcificações com margem física. A fatia escolhida
por padrão é a que contém mais voxels anotados. O fundo dos rótulos é
transparente; a sobreposição usa exatamente a mesma fatia e o mesmo recorte
da CTI. Não há limiarização HU ou geração de uma nova segmentação.

In [ ]:
calcification_ct_slice = None
calcification_label_slice = None
calcification_crop = None
calcification_z = None
spacing_xyz = None
CALCIFICATION_REFERENCE_PATH = orcascore_calcification_reference_path(
    PAIR["noncontrast"]
)
calcification_label = load_orcascore_calcification_reference(PAIR["noncontrast"])
CALCIFICATION_AVAILABLE = calcification_label is not None
if not CALCIFICATION_AVAILABLE:
    print(f"Referência de calcificação ausente: {CALCIFICATION_REFERENCE_PATH}")
else:
    noncontrast_image = load_ccta_volume(PAIR["noncontrast"])
    spacing_xyz = tuple(
        float(PAIR["noncontrast"][f"spacing_{axis}_mm"]) for axis in "xyz"
    )
    if not 0 <= CALCIFICATION_ALPHA <= 1:
        raise ValueError("CALCIFICATION_ALPHA deve estar entre 0 e 1.")
    if np.any(calcification_label):
        calcification_crop = mask_bounding_box_slices(
            calcification_label > 0, spacing_xyz, CALCIFICATION_MARGIN_MM
        )
        default_z = int(np.argmax(np.count_nonzero(calcification_label, axis=(0, 1))))
    else:
        calcification_crop = tuple(slice(0, size) for size in calcification_label.shape)
        default_z = calcification_label.shape[2] // 2
        print("Referência válida sem calcificações anotadas; exibindo a fatia central.")
    calcification_z = (
        default_z if CALCIFICATION_SLICE_INDEX is None else CALCIFICATION_SLICE_INDEX
    )
    if (
        not isinstance(calcification_z, (int, np.integer))
        or not 0 <= calcification_z < noncontrast_image.shape[2]
    ):
        raise ValueError("CALCIFICATION_SLICE_INDEX deve ser um índice axial válido.")
    xy_slices = calcification_crop[:2]
    calcification_ct_slice = noncontrast_image[*xy_slices, calcification_z].copy()
    calcification_label_slice = calcification_label[*xy_slices, calcification_z].copy()
    calcification_label_crop = calcification_label[calcification_crop]
    print(
        f"Fatia CTI: z={calcification_z} | voxels anotados nesta fatia: {np.count_nonzero(calcification_label_slice)}"
    )
    print(f"Referência: {CALCIFICATION_REFERENCE_PATH}")
    del noncontrast_image

In [ ]:
from eda.images.view.orcascore_calcification_views import show_calcification_view

CALCIFICATION_VIEW_CONFIG = {
    "calcification_available": CALCIFICATION_AVAILABLE,
    "calcification_ct_slice": calcification_ct_slice,
    "calcification_label_slice": calcification_label_slice,
    "calcification_label": calcification_label,
    "calcification_crop": calcification_crop,
    "calcification_z": calcification_z,
    "spacing_xyz": spacing_xyz,
    "figure_width_mm": FIGURE_WIDTH_MM,
    "preview_dpi": PREVIEW_DPI,
    "export_dpi": EXPORT_DPI,
    "window_level_hu": WINDOW_LEVEL_HU,
    "window_width_hu": WINDOW_WIDTH_HU,
    "calcification_alpha": CALCIFICATION_ALPHA,
    "calcification_margin_mm": CALCIFICATION_MARGIN_MM,
    "calcification_slice_index": CALCIFICATION_SLICE_INDEX,
    "calcification_reference_path": CALCIFICATION_REFERENCE_PATH,
    "exam_id": EXAM_ID,
    "subset": SUBSET,
    "pair": PAIR,
    "output_dir": OUTPUT_DIR,
}

In [ ]:
show_calcification_view("comparison", **CALCIFICATION_VIEW_CONFIG)

### CT do recorte — figura independente

In [ ]:
show_calcification_view("ct", **CALCIFICATION_VIEW_CONFIG)

### Rótulos de calcificação — figura independente

In [ ]:
show_calcification_view("label", **CALCIFICATION_VIEW_CONFIG)

### Sobreposição — figura independente

In [ ]:
show_calcification_view("overlay", **CALCIFICATION_VIEW_CONFIG)

### Superfície 3D opcional

Ative `SHOW_CALCIFICATION_3D` na configuração. A cena representa somente os
rótulos de referência na CTI, sem reorientação ou suavização das máscaras.

In [ ]:
if SHOW_CALCIFICATION_3D and CALCIFICATION_AVAILABLE:
    if np.any(calcification_label_crop):
        from IPython.display import display

        from utils.visualization.images.volume import visualize_label_map_3d

        calcification_plot = visualize_label_map_3d(
            calcification_label_crop,
            ORCASCORE_CALCIFICATION_LABEL_SPECS,
            spacing=spacing_xyz,
            opacity=0.9,
            display_plot=False,
            plot_name=f"OrCaScore {EXAM_ID}: calcificações de referência na CTI",
        )
        display(calcification_plot)
    else:
        print("Referência sem calcificações: não há superfície 3D para exibir.")